# Exercices XP 1 to 8 - Airplane Crashes Data Analysis

## Objectif : Analyse statistique complète des données d'accidents aériens avec Scipy

## Exercice 1 & 2 : Import et Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats

file_path = "/content/Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv"

if os.path.exists(file_path):
    # Ajout de l'encodage latin1 pour résoudre l'erreur de décodage
    df = pd.read_csv(file_path, encoding='latin1')
    display(df.head())
    print(f"Dimensions du dataset : {df.shape}")
else:
    print(f"Erreur : Le fichier '{file_path}' est introuvable.")

## Exercice 3 : Nettoyage et Préparation des données

In [ ]:
# Affichage des informations générales
print("Informations du DataFrame :")
print(df.info())

# Vérification des valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(df.isnull().sum())

In [ ]:
# Conversion de la colonne Date en datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Suppression des lignes avec des dates manquantes
df = df.dropna(subset=["Date"])

# Remplissage des colonnes numériques manquantes
numeric_cols = ["Aboard", "Fatalities", "Ground"]

for col in numeric_cols:
    df[col] = df[col].fillna(0)

print(f"Après nettoyage : {df.shape}")

## Exercice 4 : Feature Engineering

In [ ]:
# Création des colonnes Year et Decade
df["Year"] = df["Date"].dt.year
df["Decade"] = (df["Year"] // 10) * 10

# Création de la colonne Survivors
df["Survivors"] = df["Aboard"] - df["Fatalities"]

# Création de la colonne Survival_Rate
df["Survival_Rate"] = (df["Survivors"] / df["Aboard"]) * 100

# Affichage des premières lignes pour vérifier
display(df[["Year", "Decade", "Aboard", "Fatalities", "Survivors", "Survival_Rate"]].head(10))

## Exercice 5 : Statistiques Descriptives

In [ ]:
# Calculs globaux
total_crashes = len(df)
total_fatalities = df["Fatalities"].sum()

print(f"Total crashes: {total_crashes}")
print(f"Total fatalities: {total_fatalities}")

# Statistiques descriptives avec Scipy
print("\n=== STATISTIQUES DESCRIPTIVES (Fatalities et Survival_Rate) ===")
print(df[["Fatalities", "Survival_Rate"]].describe())

# Calculs supplémentaires avec Scipy
print("\n=== STATISTIQUES AVEC SCIPY ===")
mean_fatalities = stats.tmean(df["Fatalities"])
median_fatalities = np.median(df["Fatalities"])
std_fatalities = np.std(df["Fatalities"])
skewness = stats.skew(df["Fatalities"])
kurtosis = stats.kurtosis(df["Fatalities"])

print(f"Moyenne des fatalités (tmean): {mean_fatalities:.2f}")
print(f"Médiane des fatalités: {median_fatalities:.2f}")
print(f"Écart-type: {std_fatalities:.2f}")
print(f"Asymétrie (Skewness): {skewness:.4f}")
print(f"Kurtosis: {kurtosis:.4f}")

## Exercice 6 : Visualisations

In [ ]:
# Graphique 1 : Crashes par année
crashes_per_year = df.groupby("Year").size()

plt.figure(figsize=(14, 6))
crashes_per_year.plot()
plt.title("Nombre d'accidents aériens par année")
plt.xlabel("Année")
plt.ylabel("Nombre d'accidents")
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Graphique 2 : Crashes par décade
decade_crashes = df.groupby("Decade").size()

plt.figure(figsize=(12, 6))
decade_crashes.plot(kind='bar')
plt.title("Nombre d'accidents aériens par décade")
plt.xlabel("Décade")
plt.ylabel("Nombre d'accidents")
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Graphique 3 : Distribution des fatalités
plt.figure(figsize=(12, 6))
sns.histplot(df["Fatalities"], bins=40, kde=True)
plt.title("Distribution des fatalités")
plt.xlabel("Nombre de fatalités")
plt.ylabel("Fréquence")
plt.grid()
plt.tight_layout()
plt.show()

print("\n=== OBSERVATION ===")
print("La distribution est fortement asymétrique (skew élevé):")
print("* Beaucoup de petits accidents")
print("* Quelques catastrophes majeures")

In [ ]:
# Graphique 4 : Top 10 des localisations
top_locations = df["Location"].value_counts().head(10)

plt.figure(figsize=(12, 6))
top_locations.plot(kind='barh')
plt.title("Top 10 des localisations d'accidents aériens")
plt.xlabel("Nombre d'accidents")
plt.tight_layout()
plt.show()

print("\nTop 10 des localisations:")
print(top_locations)

In [ ]:
# Graphique 5 : Matrice de corrélation
corr = df[["Aboard", "Fatalities", "Survivors"]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

print("\nMatrice de corrélation:")
print(corr)

## Exercice 7 : Test Statistique avec Scipy - Comparaison avant/après 1980

In [ ]:
# Séparation des données avant et après 1980
before_1980 = df[df["Year"] < 1980]["Fatalities"]
after_1980 = df[df["Year"] >= 1980]["Fatalities"]

print(f"Nombre d'accidents avant 1980: {len(before_1980)}")
print(f"Nombre d'accidents après 1980: {len(after_1980)}")

# Statistiques descriptives par période
print("\n=== AVANT 1980 ===")
print(f"Moyenne: {before_1980.mean():.2f}")
print(f"Médiane: {before_1980.median():.2f}")
print(f"Écart-type: {before_1980.std():.2f}")

print("\n=== APRÈS 1980 ===")
print(f"Moyenne: {after_1980.mean():.2f}")
print(f"Médiane: {after_1980.median():.2f}")
print(f"Écart-type: {after_1980.std():.2f}")

In [ ]:
# Test t de Student indépendant avec Scipy
print("\n=== TEST T DE STUDENT (SCIPY) ===")
print("\nHypothèses:")
print("H0: Les fatalités moyennes sont identiques avant et après 1980")
print("H1: Les fatalités moyennes diffèrent avant et après 1980")

# Exécution du test
t_statistic, p_value = stats.ttest_ind(before_1980, after_1980)

print(f"\nRésultats du test:")
print(f"Statistique t: {t_statistic:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Seuil de significativité: α = 0.05")

print("\n=== INTERPRÉTATION ===")
if p_value < 0.05:
    print(f"Puisque p-value ({p_value:.6f}) < 0.05, on REJETTE H0.")
    print("CONCLUSION: Les fatalités moyennes diffèrent significativement entre les deux périodes.")
else:
    print(f"Puisque p-value ({p_value:.6f}) >= 0.05, on ne rejette pas H0.")
    print("CONCLUSION: Les fatalités moyennes ne diffèrent pas significativement entre les deux périodes.")

## Exercice 8 : Tests supplémentaires avec Scipy

In [ ]:
# Test de normalité (Shapiro-Wilk) - sur un échantillon si > 5000
print("=== TEST DE NORMALITÉ (SHAPIRO-WILK) ===")
sample_size = min(5000, len(df))
sample_fatalities = df["Fatalities"].sample(n=sample_size, random_state=42)

shapiro_stat, shapiro_p = stats.shapiro(sample_fatalities)
print(f"Statistique: {shapiro_stat:.4f}")
print(f"P-value: {shapiro_p:.6f}")

if shapiro_p < 0.05:
    print("CONCLUSION: Les données ne suivent PAS une distribution normale.")
else:
    print("CONCLUSION: Les données suivent une distribution normale.")

In [ ]:
# Test de Mann-Whitney (test non-paramétrique)
print("\n=== TEST DE MANN-WHITNEY (Test non-paramétrique) ===")
print("Utilisé car les données ne suivent pas une distribution normale")

u_statistic, u_p_value = stats.mannwhitneyu(before_1980, after_1980)
print(f"Statistique U: {u_statistic:.4f}")
print(f"P-value: {u_p_value:.6f}")

if u_p_value < 0.05:
    print("CONCLUSION: Différence significative entre les deux périodes (test non-paramétrique).")
else:
    print("CONCLUSION: Pas de différence significative entre les deux périodes (test non-paramétrique).")

In [ ]:
# Coefficient de corrélation de Spearman
print("\n=== CORRÉLATION DE SPEARMAN ===")
spearman_corr, spearman_p = stats.spearmanr(df["Aboard"], df["Fatalities"])
print(f"Coefficient de corrélation: {spearman_corr:.4f}")
print(f"P-value: {spearman_p:.6f}")
print("Interprétation: Corrélation entre le nombre de passagers à bord et les fatalités.")